# 固定步长：SDIRK2 与 SDIRK2-mr-SAV

目的：在**非零外力**下比较小步长精度、过渡区精度与大步长稳定性。每对计算使用相同初值、黏性、外力、网格、步长与物理时间。主指标为涡量相对 $L^2$ 误差，辅以速度误差。

初值为原实验的归一化多模态余弦叠加。默认读取已经验证的候选，不触发积分。计算由独立脚本完成；修改绘图不会重新计算。精确数据、验证范围和限制见同目录 `RESULTS.md`。


In [ ]:
from pathlib import Path
import json
import sys
import numpy as np
from IPython.display import display, Image, Markdown

# 支持从本 notebook 所在目录或项目根目录运行。
HERE = Path.cwd()
if not (HERE / "run.py").exists():
    HERE = HERE / "experiments" / "sdirk2_transition"
ROOT = HERE.parents[1]
sys.path.insert(0, str(HERE))
from run import Case, compare, run
from analyze import plot_comparison

# 唯一主要配置区：gamma 为 mr-SAV 参数；force 是涡量方程外力幅值。
CASE = Case(grid=256, nu=0.2, gamma=5.0, amplitude=1, force=1,
            force_k=1, initial="trig", modes=10, seed=0,
            final_time=6.0, root_selection="legacy", threads=1)
OUTPUT = ROOT / "data" / "sdirk2_transition" / "validated"
FIGURES = ROOT / "fig" / "sdirk2_transition" / "recommended"
REFERENCE_STEPS = 1536
RUN_PRODUCTION = False
RUN_REFERENCE_HALVING = False


## 显式计算入口

仅在需要重跑时打开开关。相容的已完成数据自动复用；不相容数据不会覆盖。时间步长严格为 $T/n$。四分之一时刻只在它恰好落到固定步长网格时保存；奇数步数只保存初末流场。


In [ ]:
recommended_path = HERE / "recommended.json"
if not recommended_path.exists():
    raise FileNotFoundError("缺少 recommended.json；先完成独立脚本的候选验证。")
recommended = json.loads(recommended_path.read_text())
COUNTS = recommended["counts"]
if RUN_PRODUCTION:
    compare(CASE, COUNTS, REFERENCE_STEPS, OUTPUT)
if RUN_REFERENCE_HALVING:
    run(CASE, "ETDRK4", 2 * REFERENCE_STEPS, OUTPUT)


## 从已保存数据分析

失败算例不补成终点误差。`completed` 仅表示完成了计算，不表示误差足够小；`solution_blowup` 是涡量 RMS 超过受迫连续解上界 100 倍的数值停止条件，并非声称 PDE 爆破或已产生 NaN。


In [ ]:
summary_path = ROOT / recommended["comparison"]
summary = json.loads(summary_path.read_text())
if summary["case"] != CASE.__dict__:
    raise ValueError("推荐数据与当前 CASE 不相容，请显式计算并选择相应比较文件。")
lines = ["| 步长 | SDIRK2 涡量相对误差 | mr-SAV 涡量相对误差 | 比值 |",
         "|---:|---:|---:|---:|"]
for row in sorted(summary["records"], key=lambda r: r["tau"]):
    a, b = [row["methods"][m] for m in ("IMEX_RK2", "SDIRK2_mr_SAV")]
    ea = a["errors"][-1]["omega_relative"] if a["status"] == "completed" else np.nan
    eb = b["errors"][-1]["omega_relative"] if b["status"] == "completed" else np.nan
    av = f"{ea:.6e}" if np.isfinite(ea) else a["status"]
    bv = f"{eb:.6e}" if np.isfinite(eb) else b["status"]
    rv = f"{eb/ea:.5f}" if np.isfinite(ea) and np.isfinite(eb) else "--"
    lines.append(f"| {row['tau']:.8f} | {av} | {bv} | {rv} |")
display(Markdown("\n".join(lines)))


In [ ]:
plot_comparison(summary_path, FIGURES, selected_steps=recommended["selected_steps"])
display(Image(filename=str(FIGURES / "comparison.png")))
display(Image(filename=str(FIGURES / "fields.png")))


## 解释与限制

检查连续邻近步长，而非仅挑单点；同时查看小步长误差与观测收敛阶。参考解步长减半用于估计参考误差，128² 与 256² 的同点比较用于检查空间分辨率。不同网格上的失稳阈值可以不同，不把粗网格阈值推广到其他分辨率。能量允许因外力做功而增长，因此能量上升本身不作为爆破判据。

本例只提供某一参数组的数值证据，不意味着 mr-SAV 对任意初值、外力、gamma 或步长都更准确。完整探索包含不支持预期的结果。
